# 03 — Dashboard and Genie Queries

Use these queries after running `02_incremental_refresh` at least once.

Suggested dashboard tiles:

1. Current operational demand hotspots.
2. Hourly demand trend.
3. Weekly fleet KPIs.
4. Fare anomaly review queue.

Suggested Genie space: add `gold_genie_taxi_summary`, `gold_weekly_fleet_kpis`, and `gold_fare_anomaly_review`.

In [ ]:
# Parameters
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "default")
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
print(f"Using {CATALOG}.{SCHEMA}")

## Dashboard Query 1 — Latest Hour Demand Hotspots

In Databricks SQL, paste the SQL below and create a table or map-style visualization from latitude/longitude.

In [ ]:
latest_hotspots_sql = f"""
WITH latest_hour AS (
  SELECT MAX(service_date) AS service_date
  FROM {CATALOG}.{SCHEMA}.gold_genie_taxi_summary
), latest_hour2 AS (
  SELECT g.service_date, MAX(g.service_hour) AS service_hour
  FROM {CATALOG}.{SCHEMA}.gold_genie_taxi_summary g
  JOIN latest_hour l ON g.service_date = l.service_date
  GROUP BY g.service_date
)
SELECT g.service_date,
       g.service_hour,
       g.pickup_area_latitude,
       g.pickup_area_longitude,
       g.trip_count,
       g.active_vehicles,
       g.total_revenue,
       g.demand_score
FROM {CATALOG}.{SCHEMA}.gold_genie_taxi_summary g
JOIN latest_hour2 h
  ON g.service_date = h.service_date AND g.service_hour = h.service_hour
ORDER BY g.demand_score DESC
LIMIT 25
"""
print(latest_hotspots_sql)
display(spark.sql(latest_hotspots_sql))

## Dashboard Query 2 — Hourly Demand Trend

In [ ]:
hourly_trend_sql = f"""
SELECT service_date,
       service_hour,
       SUM(trip_count) AS trips,
       SUM(total_revenue) AS revenue,
       SUM(active_vehicles) AS active_vehicle_cell_count
FROM {CATALOG}.{SCHEMA}.gold_genie_taxi_summary
GROUP BY service_date, service_hour
ORDER BY service_date, service_hour
"""
print(hourly_trend_sql)
display(spark.sql(hourly_trend_sql))

## Dashboard Query 3 — Weekly Fleet KPIs

In [ ]:
weekly_kpis_sql = f"""
SELECT week_start,
       total_clean_trips,
       active_vehicles,
       active_drivers,
       ROUND(total_revenue, 2) AS total_revenue,
       ROUND(avg_revenue_per_trip, 2) AS avg_revenue_per_trip,
       ROUND(trips_per_vehicle, 1) AS trips_per_vehicle,
       ROUND(revenue_per_vehicle, 2) AS revenue_per_vehicle,
       ROUND(card_share, 3) AS card_share,
       ROUND(cash_share, 3) AS cash_share
FROM {CATALOG}.{SCHEMA}.gold_weekly_fleet_kpis
ORDER BY week_start
"""
print(weekly_kpis_sql)
display(spark.sql(weekly_kpis_sql))

## Dashboard Query 4 — Fare Anomaly Review Queue

In [ ]:
anomaly_sql = f"""
SELECT pickup_datetime,
       medallion,
       driver_id,
       trip_distance_miles,
       trip_time_seconds,
       fare_amount,
       tip_amount,
       total_amount,
       payment_type,
       anomaly_reason,
       anomaly_score
FROM {CATALOG}.{SCHEMA}.gold_fare_anomaly_review
ORDER BY anomaly_score DESC, total_amount DESC
LIMIT 100
"""
print(anomaly_sql)
display(spark.sql(anomaly_sql))

## Genie Questions to Try

Use these exact questions in the report's Genie section. Record which worked and whether Genie needed clearer table/column comments.

- Which service date and hour had the highest number of taxi trips?
- Which pickup areas had the highest demand score in the latest available hour?
- How did total revenue and active vehicles change by week?
- What are the most common fare anomaly reasons?
- Which hours have the highest card payment share?